# IMPORT LIBRARIES & LOAD DATASET

In [3]:
import pandas as pd
import numpy as np
from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [5]:
womens_world_cup_stats = pd.read_csv(
    '/Users/gui/womens_world_cup_2023_statsbomb_finalfinal.csv'
)

# FILTER FINAL & THIRD PLACE

In [43]:
thrdplace = womens_world_cup_stats[
    womens_world_cup_stats["stage"] == "Third-place"
].copy()

thrdplace.shape

(2, 53)

In [47]:
final = womens_world_cup_stats[
    womens_world_cup_stats["stage"] == "Final"
].copy()

final.shape

(2, 53)

In [49]:
semifinalists = pd.concat(
    [thrdplace, final],
    ignore_index=True
).copy()

In [41]:
def min_max_score(series):
    if series.max() == series.min():
        return 50
    return ((series - series.min()) / (series.max() - series.min())) * 100

# CREATE THE SCORE OF THE 5 METRICS

In [51]:
semifinalists["xG_score"] = min_max_score(
    semifinalists["xG"]
)

semifinalists["pass_accuracy_score"] = min_max_score(
    semifinalists["pass_accuracy_pct"]
)

semifinalists["possession_score"] = min_max_score(
    semifinalists["possession_pct"]
)

semifinalists["goals_scored_score"] = min_max_score(
    semifinalists["goals_for"]
)

semifinalists["goals_conceded_score"] = (
    100 - min_max_score(semifinalists["goals_against"])
)

# PERFORMANCE SCORE

In [53]:
semifinalists["performance_score"] = (
    semifinalists["xG_score"] * 0.25 +
    semifinalists["pass_accuracy_score"] * 0.25 +
    semifinalists["possession_score"] * 0.10 +
    semifinalists["goals_scored_score"] * 0.25 +
    semifinalists["goals_conceded_score"] * 0.15
)

In [55]:
semifinalists["performance_score"] = semifinalists["performance_score"].round(1)

# FINAL TABLE

In [57]:
performance_table = semifinalists[
    [
        "team",
        "xG",
        "pass_accuracy_pct",
        "possession_pct",
        "goals_for",
        "goals_against",
        "performance_score"
    ]
].copy()

performance_table = performance_table.sort_values(
    "performance_score",
    ascending=False
).reset_index(drop=True)



performance_table.insert(
    0,
    "Rank",
    range(1, len(performance_table) + 1)
)


performance_table["xG"] = (
    performance_table["xG"].round(1)
)

performance_table["pass_accuracy_pct"] = (
    performance_table["pass_accuracy_pct"].round(1)
)

performance_table["possession_pct"] = (
    performance_table["possession_pct"].round(1)
)


performance_table = performance_table.rename(columns={
    "team": "Team",
    "xG": "xG (%)",
    "pass_accuracy_pct": "Pass Accuracy (%)",
    "possession_pct": "Possession (%)",
    "goals_for": "Goals Scored",
    "goals_against": "Goals Conceded",
    "performance_score": "Performance Score"
})



performance_table

,Rank,Team,xG (%),Pass Accuracy (%),Possession (%),Goals Scored,Goals Conceded,Performance Score
0,1,Spain Women's,2.2,81.1,55.7,1,0,87.5
1,2,Sweden Women's,1.6,74.3,47.2,2,0,70.6
2,3,England Women's,0.5,71.8,44.3,0,1,14.8
3,4,Australia Women's,0.7,67.9,52.8,0,2,11.4


## Table 2: Score Breakdown

In [60]:
# ==========================================
# SCORE BREAKDOWN
# ==========================================

score_breakdown = semifinalists[
    [
        "team",
        "xG_score",
        "pass_accuracy_score",
        "possession_score",
        "goals_scored_score",
        "goals_conceded_score",
        "performance_score"
    ]
].copy()



score_breakdown = score_breakdown.sort_values(
    "performance_score",
    ascending=False
).reset_index(drop=True)



score_breakdown.insert(
    0,
    "Rank",
    range(1, len(score_breakdown) + 1)
)



score_columns = [
    "xG_score",
    "pass_accuracy_score",
    "possession_score",
    "goals_scored_score",
    "goals_conceded_score",
    "performance_score"
]

score_breakdown[score_columns] = (
    score_breakdown[score_columns].round(1)
)



score_breakdown = score_breakdown.rename(columns={
    "team": "Team",
    "xG": "xG (25%)",
    "pass_accuracy_score": "Pass Accuracy Score (25%)",
    "possession_score": "Possession Score (10%)",
    "goals_scored_score": "Goals Scored Score (25%)",
    "goals_conceded_score": "Goals Conceded Score (15%)",
    "performance_score": "Performance Score"
})



score_breakdown

,Rank,Team,xG_score,Pass Accuracy Score (25%),Possession Score (10%),Goals Scored Score (25%),Goals Conceded Score (15%),Performance Score
0,1,Spain Women's,100.0,100.0,100.0,50.0,100.0,87.5
1,2,Sweden Women's,63.8,48.4,25.3,100.0,100.0,70.6
2,3,England Women's,0.0,29.4,0.0,0.0,50.0,14.8
3,4,Australia Women's,15.9,0.0,74.7,0.0,0.0,11.4


# CONCLUSION KEY FINDINGS

Spain achieved the highest Performance Score (87.5), clearly standing out through their combination of xG, possession, passing accuracy and a positive result. Sweden followed with 70.6, benefiting from strong finishing and defensive solidity, scoring twice and keeping a clean sheet despite having less possession.

England and Australia recorded much lower scores, mainly due to their limited attacking output. England's 0.5 xG and failure to score significantly reduced their score, while Australia had the lowest xG and pass accuracy and conceded twice. Overall, the results highlight Spain's consistency across the different performance dimensions.